In [1]:
%matplotlib widget
%matplotlib widget
import os
from pathlib import Path
import time
import torch
import numpy as np
import math
import gc
from functools import partial
from dataset_alt import Dataset, load_dataframes_from_folder, reverse_normalization
from torch.utils.data import DataLoader
from transformer_zerostep import GPTConfig, GPT, warmup_cosine_lr
from transformer_zerostep_alt import GPT_alt
import argparse
import warnings
import matplotlib.pyplot as plt
import onnxruntime as rt
from transformer_zerostep_IF import GPT_if
import onnx
import copy


# set figure parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams['axes.labelsize']=14
plt.rcParams['xtick.labelsize']=11
plt.rcParams['ytick.labelsize']=11
plt.rcParams['axes.grid']=True
plt.rcParams['axes.xmargin']=0

In [2]:
# # Overall settings
# out_dir = "out"

# model_name = "new_delay_h10_10k.pt"
# # model_name = "model_high_speed.pt"

# current_path = os.getcwd().split("in-context-bldc")[0]
# data_path = os.path.join(current_path,"in-context-bldc", "data")

# folder = "simulated/50_percent_control/validation"
# # folder = "CL_experiments_double_sensor_control/test/inertia13"
# folder_path = os.path.join(data_path, folder)

# # Compute settings
# cuda_device = "cuda:0"
# no_cuda = True
# threads = 10
# compile = False

# # Configure compute
# torch.set_num_threads(threads) 
# use_cuda = not no_cuda and torch.cuda.is_available()
# device_name  = cuda_device if use_cuda else "cpu"
# device = torch.device(device_name)
# device_type = 'cuda' if 'cuda' in device_name else 'cpu' # for later use in torch.autocast
# torch.set_float32_matmul_precision("high")
# print(torch.cuda.is_available())
# # Create out dir
# out_dir = Path(out_dir)
# exp_data = torch.load(out_dir/model_name, map_location=device, weights_only=False)
# seq_len = exp_data["cfg"].seq_len
# nx = exp_data["cfg"].nx
# exp_data["iter_num"]
# print(seq_len)
# print(exp_data["iter_num"])
# print(exp_data['best_val_loss'])
# print(exp_data["cfg"])
# print(exp_data["cfg"].lr)
# print(exp_data["train_time"]/3600)
# print(exp_data["model_args"])

In [3]:
# generate the model
# model_args = exp_data["model_args"]
# gptconf = GPTConfig(**model_args)
# model = GPT_alt(gptconf).to(device)
model_args = dict(n_layer=1, n_head=1, n_embd=1, n_x=1, n_y=1, n_u=8, block_size=10,
                      bias=False, dropout=0.0)
gptconf = GPTConfig(**model_args)
model = GPT_alt(gptconf).to('cpu')
# print(model.get_num_params())
model.eval()
torch.save(model, "empty_model.pt")

model = torch.load("empty_model.pt", weights_only=False)


number of parameters: 0.00M


In [4]:
input_test_a = torch.zeros(1,10,8)

print(model(input_test_a))
print(model(input_test_a).size())
print()

tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])
tensor([[[0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.]]], grad_fn=<ViewBackward0>)
tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])
torch.Size([1, 10, 1])



In [5]:
model_name_traced = "empty_model_traced.pt"
model.eval()

traced_model = torch.jit.trace(model, input_test_a)
torch.jit.save(traced_model, model_name_traced)
model_jit = torch.jit.load(model_name_traced)


tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])
tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])
tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])
